In [1]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ MERGE v2.5 — Combine results + Compare MSE vs CVaR Curriculum             ║
# ║                                                                            ║
# ║ Loads: results_v25_GBM.json + results_v25_Heston.json +                    ║
# ║        results_v25_SBTS.json + old results (v2.2)                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import json, os
import numpy as np

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_FOLDER = '/content/drive/MyDrive/PHD_SBTS'

# ═══════════════════════════════════════════════════════════════
#  LOAD ALL RESULTS
# ═══════════════════════════════════════════════════════════════
print("=" * 70)
print("LOADING ALL RESULTS")
print("=" * 70)

# v2.5 curriculum results
v25_all = {}
for ds in ['GBM', 'Heston', 'SBTS']:
    fpath = os.path.join(DRIVE_FOLDER, f'results_v25_{ds}.json')
    if os.path.exists(fpath):
        with open(fpath, 'r') as f:
            data = json.load(f)
        print(f"  v2.5 {ds}: {len(data)} runs")
        v25_all.update(data)
    else:
        print(f"  v2.5 {ds}: ❌ not found")

print(f"  v2.5 total: {len(v25_all)} / 180")

# Old v2.2 results (MSE + CVaR from scratch)
v22_all = {}
for ds in ['GBM', 'Heston', 'SBTS']:
    fpath = os.path.join(DRIVE_FOLDER, f'results_{ds}.json')
    if os.path.exists(fpath):
        with open(fpath, 'r') as f:
            data = json.load(f)
        v22_all.update(data)

print(f"  v2.2 total: {len(v22_all)} runs")

# Verify completeness
OPTION_NAMES = ['basket_asian_call', 'asian_worst_of_put']
KAPPA_LEVELS = [0.95, 1.00, 1.05]
DS_LIST = ['GBM', 'Heston', 'SBTS']
N_SEEDS = 10

missing = []
for ds in DS_LIST:
    for opt in OPTION_NAMES:
        for kappa in KAPPA_LEVELS:
            for seed in range(N_SEEDS):
                key = f"{ds}_v25_{opt}_k{kappa:.2f}_s{seed}"
                if key not in v25_all:
                    missing.append(key)

if missing:
    print(f"\n  ⚠️ {len(missing)} v2.5 runs missing:")
    for m in missing[:10]:
        print(f"    {m}")
    if len(missing) > 10:
        print(f"    ... and {len(missing)-10} more")
else:
    print(f"\n  ✅ All 180 v2.5 runs complete!")

# Save merged
merged_path = os.path.join(DRIVE_FOLDER, 'results_v25_merged.json')
with open(merged_path, 'w') as f:
    json.dump(v25_all, f, indent=2)
print(f"  Saved: {merged_path}")

# ═══════════════════════════════════════════════════════════════
#  ANALYSIS: MSE (from v2.5) vs CVaR curriculum (from v2.5)
#            vs CVaR from scratch (from v2.2)
# ═══════════════════════════════════════════════════════════════

def get_hist(result, phase, period, metric):
    """Extract metric from v2.5 result structure."""
    return result.get(phase, {}).get('historical', {}).get(period, {}).get(metric, float('nan'))

def get_test(result, phase, metric):
    return result.get(phase, {}).get('test', {}).get(metric, float('nan'))

# For v2.2 old format
def get_hist_v22(result, period, metric):
    return result.get('historical', {}).get(period, {}).get(metric, float('nan'))

def get_test_v22(result, metric):
    return result.get('test', {}).get(metric, float('nan'))

hist_periods = ['COVID_2020', 'PostCOVID_2021_22', 'Normal_2023_24']
short = {'COVID_2020': 'COVID', 'PostCOVID_2021_22': 'PostCOV', 'Normal_2023_24': 'Normal'}

print(f"\n\n{'═' * 70}")
print("ANALYSIS — MSE vs CVaR Curriculum vs CVaR from Scratch")
print(f"{'═' * 70}")

for opt in OPTION_NAMES:
    for kappa in KAPPA_LEVELS:
        print(f"\n{'━' * 70}")
        print(f"  {opt} κ={kappa:.2f}")
        print(f"{'━' * 70}")

        for metric_name, metric_key in [('Std', 'std'), ('CVaR95', 'cvar95')]:
            print(f"\n  {metric_name}:")
            print(f"  {'Generator':<10s} {'MSE':>8s} {'CVaR v2.2':>10s} {'CVaR v2.5':>10s} "
                  f"{'v2.5 vs MSE':>12s} {'v2.5 vs v2.2':>12s}")
            print(f"  {'─'*66}")

            for ds in DS_LIST:
                # MSE from v2.5 (Phase 1)
                mse_vals = []
                cvar_v25_vals = []
                cvar_v22_vals = []

                for seed in range(N_SEEDS):
                    # v2.5
                    v25_key = f"{ds}_v25_{opt}_k{kappa:.2f}_s{seed}"
                    if v25_key in v25_all:
                        r = v25_all[v25_key]
                        # Grand avg across periods
                        mse_period_vals = [get_hist(r, 'mse', p, metric_key) for p in hist_periods]
                        cvar_period_vals = [get_hist(r, 'cvar', p, metric_key) for p in hist_periods]
                        mse_period_vals = [v for v in mse_period_vals if not np.isnan(v)]
                        cvar_period_vals = [v for v in cvar_period_vals if not np.isnan(v)]
                        if mse_period_vals:
                            mse_vals.append(np.mean(mse_period_vals))
                        if cvar_period_vals:
                            cvar_v25_vals.append(np.mean(cvar_period_vals))

                    # v2.2 CVaR from scratch
                    v22_key = f"{ds}_CVaR95_{opt}_k{kappa:.2f}_s{seed}"
                    if v22_key in v22_all:
                        r22 = v22_all[v22_key]
                        v22_period_vals = [get_hist_v22(r22, p, metric_key) for p in hist_periods]
                        v22_period_vals = [v for v in v22_period_vals if not np.isnan(v)]
                        if v22_period_vals:
                            cvar_v22_vals.append(np.mean(v22_period_vals))

                m_mse = np.mean(mse_vals) if mse_vals else float('nan')
                m_v25 = np.mean(cvar_v25_vals) if cvar_v25_vals else float('nan')
                m_v22 = np.mean(cvar_v22_vals) if cvar_v22_vals else float('nan')

                gap_mse = (m_v25/m_mse - 1)*100 if not np.isnan(m_mse) and not np.isnan(m_v25) else float('nan')
                gap_v22 = (m_v25/m_v22 - 1)*100 if not np.isnan(m_v22) and not np.isnan(m_v25) else float('nan')

                print(f"  {ds:<10s} {m_mse:8.4f} {m_v22:10.4f} {m_v25:10.4f} "
                      f"{gap_mse:+11.1f}% {gap_v22:+11.1f}%")


# ═══════════════════════════════════════════════════════════════
#  GRAND SUMMARY
# ═══════════════════════════════════════════════════════════════
print(f"\n\n{'═' * 70}")
print("GRAND SUMMARY — All 6 configs averaged")
print(f"{'═' * 70}")

for metric_name, metric_key in [('Std', 'std'), ('CVaR95', 'cvar95'),
                                 ('CVaR99', 'cvar99'), ('Max', 'max')]:
    all_mse, all_v25, all_v22 = [], [], []

    for ds in DS_LIST:
        for opt in OPTION_NAMES:
            for kappa in KAPPA_LEVELS:
                for seed in range(N_SEEDS):
                    v25_key = f"{ds}_v25_{opt}_k{kappa:.2f}_s{seed}"
                    if v25_key in v25_all:
                        r = v25_all[v25_key]
                        mse_pv = [get_hist(r, 'mse', p, metric_key) for p in hist_periods]
                        cvar_pv = [get_hist(r, 'cvar', p, metric_key) for p in hist_periods]
                        mse_pv = [v for v in mse_pv if not np.isnan(v)]
                        cvar_pv = [v for v in cvar_pv if not np.isnan(v)]
                        if mse_pv: all_mse.append(np.mean(mse_pv))
                        if cvar_pv: all_v25.append(np.mean(cvar_pv))

                    v22_key = f"{ds}_CVaR95_{opt}_k{kappa:.2f}_s{seed}"
                    if v22_key in v22_all:
                        r22 = v22_all[v22_key]
                        v22_pv = [get_hist_v22(r22, p, metric_key) for p in hist_periods]
                        v22_pv = [v for v in v22_pv if not np.isnan(v)]
                        if v22_pv: all_v22.append(np.mean(v22_pv))

    m_mse = np.mean(all_mse) if all_mse else float('nan')
    m_v25 = np.mean(all_v25) if all_v25 else float('nan')
    m_v22 = np.mean(all_v22) if all_v22 else float('nan')

    gap_mse = (m_v25/m_mse - 1)*100 if not np.isnan(m_mse) and not np.isnan(m_v25) else float('nan')
    gap_v22 = (m_v25/m_v22 - 1)*100 if not np.isnan(m_v22) and not np.isnan(m_v25) else float('nan')

    print(f"  {metric_name:<8s}  MSE={m_mse:.4f}  CVaR_scratch={m_v22:.4f}  "
          f"CVaR_curric={m_v25:.4f}  v2.5vsMSE={gap_mse:+.1f}%  v2.5vsV22={gap_v22:+.1f}%")

# ═══════════════════════════════════════════════════════════════
#  WINNER TABLE
# ═══════════════════════════════════════════════════════════════
print(f"\n\n{'═' * 70}")
print("WINNER TABLE — Which loss wins per config?")
print(f"{'═' * 70}")

print(f"\n  Based on Std (average hedging quality):")
print(f"  {'Option':<25s} {'κ':>5s} {'GBM':>8s} {'Heston':>8s} {'SBTS':>8s}")
print(f"  {'─'*58}")

for opt in OPTION_NAMES:
    for kappa in KAPPA_LEVELS:
        line = f"  {opt:<25s} {kappa:5.2f}"
        for ds in DS_LIST:
            mse_vals, v25_vals = [], []
            for seed in range(N_SEEDS):
                v25_key = f"{ds}_v25_{opt}_k{kappa:.2f}_s{seed}"
                if v25_key in v25_all:
                    r = v25_all[v25_key]
                    mv = [get_hist(r, 'mse', p, 'std') for p in hist_periods]
                    cv = [get_hist(r, 'cvar', p, 'std') for p in hist_periods]
                    mv = [v for v in mv if not np.isnan(v)]
                    cv = [v for v in cv if not np.isnan(v)]
                    if mv: mse_vals.append(np.mean(mv))
                    if cv: v25_vals.append(np.mean(cv))
            m = np.mean(mse_vals) if mse_vals else float('nan')
            v = np.mean(v25_vals) if v25_vals else float('nan')
            winner = 'CVaR' if v < m else 'MSE'
            line += f" {winner:>8s}"
        print(line)

print(f"\n  Based on CVaR95 (tail risk protection):")
print(f"  {'Option':<25s} {'κ':>5s} {'GBM':>8s} {'Heston':>8s} {'SBTS':>8s}")
print(f"  {'─'*58}")

for opt in OPTION_NAMES:
    for kappa in KAPPA_LEVELS:
        line = f"  {opt:<25s} {kappa:5.2f}"
        for ds in DS_LIST:
            mse_vals, v25_vals = [], []
            for seed in range(N_SEEDS):
                v25_key = f"{ds}_v25_{opt}_k{kappa:.2f}_s{seed}"
                if v25_key in v25_all:
                    r = v25_all[v25_key]
                    mv = [get_hist(r, 'mse', p, 'cvar95') for p in hist_periods]
                    cv = [get_hist(r, 'cvar', p, 'cvar95') for p in hist_periods]
                    mv = [v for v in mv if not np.isnan(v)]
                    cv = [v for v in cv if not np.isnan(v)]
                    if mv: mse_vals.append(np.mean(mv))
                    if cv: v25_vals.append(np.mean(cv))
            m = np.mean(mse_vals) if mse_vals else float('nan')
            v = np.mean(v25_vals) if v25_vals else float('nan')
            winner = 'CVaR' if v < m else 'MSE'
            line += f" {winner:>8s}"
        print(line)


print(f"\n{'═' * 70}")
print("DONE")
print(f"{'═' * 70}")


Mounted at /content/drive
LOADING ALL RESULTS
  v2.5 GBM: 60 runs
  v2.5 Heston: 60 runs
  v2.5 SBTS: 60 runs
  v2.5 total: 180 / 180
  v2.2 total: 360 runs

  ✅ All 180 v2.5 runs complete!
  Saved: /content/drive/MyDrive/PHD_SBTS/results_v25_merged.json


══════════════════════════════════════════════════════════════════════
ANALYSIS — MSE vs CVaR Curriculum vs CVaR from Scratch
══════════════════════════════════════════════════════════════════════

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  basket_asian_call κ=0.95
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Std:
  Generator       MSE  CVaR v2.2  CVaR v2.5  v2.5 vs MSE v2.5 vs v2.2
  ──────────────────────────────────────────────────────────────────
  GBM          0.0110     0.0098     0.0098       -10.5%        +0.3%
  Heston       0.0106     0.0095     0.0096        -8.8%        +1.2%
  SBTS         0.0100     0.0096     0.0091        -9.4%        -5.0%

  CVaR95:
  Genera

In [1]:
import numpy as np

data = np.load('/content/drive/MyDrive/PHD_SBTS/historical_test_paths.npz', allow_pickle=True)

print("Keys:", list(data.keys()))
print()

# Period names
if 'period_names' in data:
    periods = data['period_names']
    print("Periods:", periods)
    print()

# Check each period
for key in sorted(data.keys()):
    arr = data[key]
    print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")
    if 'S_norm' in key:
        print(f"    → {arr.shape[0]} paths × {arr.shape[1]} time steps × {arr.shape[2]} assets")
        print(f"    → Window size: {arr.shape[1]-1} trading days")

        # Check if paths overlap
        if arr.shape[0] > 1:
            # Compare starting points of consecutive paths
            starts = arr[:5, 0, :]  # first 5 paths, time 0
            print(f"    → First 5 starting values (asset 1): {starts[:, 0]}")
            print(f"    → All start at 1.0? {np.allclose(starts, 1.0)}")

print("\n" + "="*50)
print("SUMMARY")
print("="*50)
for key in sorted(data.keys()):
    if 'S_norm' in key:
        n_paths = data[key].shape[0]
        n_steps = data[key].shape[1] - 1
        period = key.replace('S_norm_', '')
        print(f"  {period}: {n_paths} paths of {n_steps} days")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/PHD_SBTS/historical_test_paths.npz'

In [3]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ FULL ANALYSIS v2 — Fixes 2 data issues:                                   ║
# ║   1. CVaR95 ≈ 0 (or negative) in Normal period → flag, use absolute       ║
# ║   2. % calculation blows up when base ≈ 0 → cap or use Δ instead          ║
# ║                                                                            ║
# ║ Rule: if |base| < ZERO_THRESHOLD → report absolute Δ, not %              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import json, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_FOLDER = '/content/drive/MyDrive/PHD_SBTS'
FIG_FOLDER = os.path.join(DRIVE_FOLDER, 'figures_v25')
os.makedirs(FIG_FOLDER, exist_ok=True)

# ═══════════════════════════════════════════════════════════════
#  THRESHOLD — below this, % is meaningless
# ═══════════════════════════════════════════════════════════════
ZERO_THRESHOLD = 0.005  # if |metric| < 0.005, treat as ≈0

def safe_pct(val, base):
    """Return % change, or None if base too small."""
    if abs(base) < ZERO_THRESHOLD:
        return None
    return (val / base - 1) * 100

def fmt_pct(pct):
    """Format % or '≈0' if None."""
    if pct is None:
        return "   ≈0   "
    return f"{pct:+7.1f}%"

def fmt_val(v):
    """Format metric value, flagging near-zero."""
    if abs(v) < ZERO_THRESHOLD:
        return f"{'≈0':>8s}"
    return f"{v:8.4f}"

# ═══════════════════════════════════════════════════════════════
#  LOAD
# ═══════════════════════════════════════════════════════════════
print("=" * 70)
print("LOADING ALL RESULTS")
print("=" * 70)

v25 = {}
for ds in ['GBM', 'Heston', 'SBTS']:
    fpath = os.path.join(DRIVE_FOLDER, f'results_v25_{ds}.json')
    if os.path.exists(fpath):
        with open(fpath, 'r') as f:
            data = json.load(f)
        v25.update(data)
        print(f"  v2.5 {ds}: {len(data)} runs")

v22 = {}
for ds in ['GBM', 'Heston', 'SBTS']:
    fpath = os.path.join(DRIVE_FOLDER, f'results_{ds}.json')
    if os.path.exists(fpath):
        with open(fpath, 'r') as f:
            data = json.load(f)
        v22.update(data)

print(f"\n  v2.5 total: {len(v25)} (MSE + CVaR curriculum)")
print(f"  v2.2 total: {len(v22)} (MSE + CVaR scratch)")

# ═══════════════════════════════════════════════════════════════
#  CONFIG
# ═══════════════════════════════════════════════════════════════
DS_LIST = ['GBM', 'Heston', 'SBTS']
OPTION_NAMES = ['basket_asian_call', 'asian_worst_of_put']
KAPPA_LEVELS = [0.95, 1.00, 1.05]
N_SEEDS = 10
PERIODS = ['COVID_2020', 'PostCOVID_2021_22', 'Normal_2023_24']
SHORT_P = {'COVID_2020': 'COVID', 'PostCOVID_2021_22': 'PostCOV', 'Normal_2023_24': 'Normal'}
LOSS_TYPES = ['MSE', 'CVaR_scratch', 'CVaR_curriculum']
LOSS_LABELS = ['MSE', 'CVaR (scratch)', 'CVaR (curriculum)']
LOSS_COLORS = ['#2196F3', '#FF5722', '#4CAF50']
LOSS_SHORT = ['MSE', 'CVaR', 'Curric']

# ═══════════════════════════════════════════════════════════════
#  EXTRACT
# ═══════════════════════════════════════════════════════════════
def collect(ds, opt, kappa, lt, period, metric):
    vals = []
    for seed in range(N_SEEDS):
        v = None
        if lt == 'MSE':
            key = f"{ds}_v25_{opt}_k{kappa:.2f}_s{seed}"
            if key in v25:
                v = v25[key].get('mse', {}).get('historical', {}).get(period, {}).get(metric)
        elif lt == 'CVaR_scratch':
            key = f"{ds}_CVaR95_{opt}_k{kappa:.2f}_s{seed}"
            if key in v22:
                v = v22[key].get('historical', {}).get(period, {}).get(metric)
        elif lt == 'CVaR_curriculum':
            key = f"{ds}_v25_{opt}_k{kappa:.2f}_s{seed}"
            if key in v25:
                v = v25[key].get('cvar', {}).get('historical', {}).get(period, {}).get(metric)
        if v is not None:
            vals.append(v)
    return vals

def grand_by_period(lt, period, metric):
    av = []
    for ds in DS_LIST:
        for opt in OPTION_NAMES:
            for kappa in KAPPA_LEVELS:
                av.extend(collect(ds, opt, kappa, lt, period, metric))
    return np.mean(av) if av else 0


# ═══════════════════════════════════════════════════════════════
#  TABLE 1: GRAND SUMMARY per period (absolute values)
# ═══════════════════════════════════════════════════════════════
print(f"\n\n{'═' * 80}")
print("TABLE 1: GRAND SUMMARY — Absolute Values (all 18 configs averaged)")
print(f"{'═' * 80}")

for mk, mn in [('std','Std'), ('cvar95','CVaR95'), ('cvar99','CVaR99'), ('max','Max')]:
    print(f"\n  {mn}:")
    print(f"  {'Loss':<22s}", end="")
    for p in PERIODS:
        print(f" {SHORT_P[p]:>8s}", end="")
    print(f" {'Grand':>8s}")
    print(f"  {'─'*56}")

    for lt, ll in zip(LOSS_TYPES, LOSS_LABELS):
        line = f"  {ll:<22s}"
        pgs = []
        for p in PERIODS:
            m = grand_by_period(lt, p, mk)
            pgs.append(m)
            line += f" {fmt_val(m)}"
        g = np.mean(pgs)
        line += f" {g:8.4f}"
        print(line)


# ═══════════════════════════════════════════════════════════════
#  TABLE 2: CVaR CURRICULUM vs MSE — Absolute Δ + Safe %
# ═══════════════════════════════════════════════════════════════
print(f"\n\n{'═' * 80}")
print("TABLE 2: CVaR CURRICULUM vs MSE — Absolute Δ and Safe %")
print(f"  (% shown only when |MSE base| ≥ {ZERO_THRESHOLD}; otherwise '≈0')")
print(f"{'═' * 80}")

for mk, mn in [('std','Std'), ('cvar95','CVaR95')]:
    print(f"\n  {mn}:")
    print(f"  {'Option':<12s} {'κ':>5s} {'Gen':>7s}", end="")
    for p in PERIODS:
        print(f" {SHORT_P[p]+' Δ':>9s} {SHORT_P[p]+' %':>8s}", end="")
    print(f" {'Grand Δ':>8s} {'Grand%':>8s}")
    print(f"  {'─'*96}")

    for opt in OPTION_NAMES:
        for kappa in KAPPA_LEVELS:
            for ds in DS_LIST:
                os_ = 'Basket' if 'basket' in opt else 'WoP'
                line = f"  {os_:<12s} {kappa:5.2f} {ds:>7s}"
                deltas = []
                pcts = []
                for p in PERIODS:
                    mv = collect(ds, opt, kappa, 'MSE', p, mk)
                    cv = collect(ds, opt, kappa, 'CVaR_curriculum', p, mk)
                    if mv and cv:
                        m_mse = np.mean(mv)
                        m_cur = np.mean(cv)
                        delta = m_cur - m_mse
                        pct = safe_pct(m_cur, m_mse)
                        deltas.append(delta)
                        if pct is not None:
                            pcts.append(pct)
                        line += f" {delta:+8.4f}  {fmt_pct(pct)}"
                    else:
                        line += f" {'—':>9s} {'—':>8s}"

                # Grand
                if deltas:
                    gd = np.mean(deltas)
                    gp = np.mean(pcts) if pcts else None
                    line += f" {gd:+7.4f}  {fmt_pct(gp)}"
                print(line)


# ═══════════════════════════════════════════════════════════════
#  TABLE 3: WINNER COUNT per period
# ═══════════════════════════════════════════════════════════════
print(f"\n\n{'═' * 80}")
print("TABLE 3: WINNER COUNT per period (18 configs each)")
print(f"{'═' * 80}")

for mk, mn in [('std','Std'), ('cvar95','CVaR95')]:
    print(f"\n  {mn}:")
    print(f"  {'Period':<12s} {'MSE':>6s} {'CVaR':>6s} {'Curric':>6s} {'Total':>6s}")
    print(f"  {'─'*40}")

    tc = {'MSE': 0, 'CVaR': 0, 'Curric': 0}
    gt = 0
    for p in PERIODS:
        counts = {'MSE': 0, 'CVaR': 0, 'Curric': 0}
        for ds in DS_LIST:
            for opt in OPTION_NAMES:
                for kappa in KAPPA_LEVELS:
                    bv, bl = float('inf'), 'MSE'
                    for lt, ls in zip(LOSS_TYPES, LOSS_SHORT):
                        vals = collect(ds, opt, kappa, lt, p, mk)
                        m = np.mean(vals) if vals else float('inf')
                        if m < bv:
                            bv, bl = m, ls
                    counts[bl] += 1; tc[bl] += 1; gt += 1
        print(f"  {SHORT_P[p]:<12s} {counts['MSE']:6d} {counts['CVaR']:6d} "
              f"{counts['Curric']:6d} {18:6d}")
    print(f"  {'TOTAL':<12s} {tc['MSE']:6d} {tc['CVaR']:6d} {tc['Curric']:6d} {gt:6d}")


# ═══════════════════════════════════════════════════════════════
#  TABLE 4: THESIS-READY SUMMARY TABLE
# ═══════════════════════════════════════════════════════════════
print(f"\n\n{'═' * 80}")
print("TABLE 4: THESIS-READY — Grand Average (all configs, absolute)")
print(f"{'═' * 80}")

print(f"\n  {'Metric':<10s} {'MSE':>8s} {'CVaR scr':>10s} {'CVaR cur':>10s} "
      f"{'Δ(cur-MSE)':>10s} {'%':>8s}")
print(f"  {'─'*58}")

for mk, mn in [('std','Std'), ('cvar95','CVaR95'), ('cvar99','CVaR99'), ('max','Max')]:
    vals_by_lt = {}
    for lt in LOSS_TYPES:
        all_v = []
        for ds in DS_LIST:
            for opt in OPTION_NAMES:
                for kappa in KAPPA_LEVELS:
                    for p in PERIODS:
                        all_v.extend(collect(ds, opt, kappa, lt, p, mk))
        vals_by_lt[lt] = np.mean(all_v) if all_v else 0

    m = vals_by_lt['MSE']
    s = vals_by_lt['CVaR_scratch']
    c = vals_by_lt['CVaR_curriculum']
    delta = c - m
    pct = safe_pct(c, m)
    print(f"  {mn:<10s} {m:8.4f} {s:10.4f} {c:10.4f} {delta:+9.4f}  {fmt_pct(pct)}")


# ═══════════════════════════════════════════════════════════════
#  TABLE 5: PER-PERIOD THESIS TABLE (COVID focus)
# ═══════════════════════════════════════════════════════════════
print(f"\n\n{'═' * 80}")
print("TABLE 5: PER-PERIOD BREAKDOWN — Thesis Format")
print(f"  Note: % shown only when meaningful (|base| ≥ {ZERO_THRESHOLD})")
print(f"{'═' * 80}")

for mk, mn in [('std','Std'), ('cvar95','CVaR95')]:
    print(f"\n  {mn}:")
    print(f"  {'Period':<12s} {'MSE':>8s} {'CVaR scr':>10s} {'CVaR cur':>10s} "
          f"{'Δ(cur-MSE)':>10s} {'%':>8s} {'Note':>12s}")
    print(f"  {'─'*74}")

    for p in PERIODS:
        m = grand_by_period('MSE', p, mk)
        s = grand_by_period('CVaR_scratch', p, mk)
        c = grand_by_period('CVaR_curriculum', p, mk)
        delta = c - m
        pct = safe_pct(c, m)

        note = ""
        if abs(m) < ZERO_THRESHOLD and abs(c) < ZERO_THRESHOLD:
            note = "both ≈ 0"
        elif abs(m) < ZERO_THRESHOLD:
            note = "MSE ≈ 0"

        print(f"  {SHORT_P[p]:<12s} {fmt_val(m)} {fmt_val(s):>10s} {fmt_val(c):>10s} "
              f"{delta:+9.4f}  {fmt_pct(pct)} {note:>12s}")

    # Grand
    m = np.mean([grand_by_period('MSE', p, mk) for p in PERIODS])
    s = np.mean([grand_by_period('CVaR_scratch', p, mk) for p in PERIODS])
    c = np.mean([grand_by_period('CVaR_curriculum', p, mk) for p in PERIODS])
    delta = c - m
    pct = safe_pct(c, m)
    print(f"  {'Grand':<12s} {m:8.4f} {s:10.4f} {c:10.4f} {delta:+9.4f}  {fmt_pct(pct)}")


# ═══════════════════════════════════════════════════════════════
#  FIGURES (same 5, with near-zero handling)
# ═══════════════════════════════════════════════════════════════
print(f"\n\n{'═' * 80}")
print("GENERATING FIGURES")
print(f"{'═' * 80}")

plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

SHORT_PERIODS_DISPLAY = ['COVID\n2020', 'Post-COVID\n2021-22', 'Normal\n2023-24']


# ── FIG 1: Grand Summary (4 metrics × 3 losses × 3 periods) ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Hedging Performance by Historical Period\n'
             '(grand average: 3 generators × 2 options × 3 strikes × 10 seeds)',
             fontsize=13, fontweight='bold')

for ai, (mk, mn) in enumerate(
    [('std','Std'), ('cvar95','CVaR₉₅'), ('cvar99','CVaR₉₉'), ('max','Max Loss')]):
    ax = axes[ai//2, ai%2]
    x = np.arange(len(PERIODS)); w = 0.25
    for i, (lt, ll, col) in enumerate(zip(LOSS_TYPES, LOSS_LABELS, LOSS_COLORS)):
        vals = [max(grand_by_period(lt, p, mk), 0) for p in PERIODS]  # clip negative
        bars = ax.bar(x + i*w, vals, w, label=ll, color=col, alpha=0.85,
                       edgecolor='white', linewidth=0.5)
        for b, v in zip(bars, vals):
            label = f'{v:.4f}' if v >= ZERO_THRESHOLD else '≈0'
            ax.text(b.get_x()+b.get_width()/2, b.get_height() + 0.0005,
                    label, ha='center', va='bottom', fontsize=7)
    ax.set_title(mn); ax.set_xticks(x+w)
    ax.set_xticklabels(SHORT_PERIODS_DISPLAY)
    ax.set_ylabel(mn.split('(')[0].strip())
    ax.set_ylim(bottom=0)
    if ai == 0: ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fp = os.path.join(FIG_FOLDER, 'fig1_grand_by_period.pdf')
plt.savefig(fp); plt.savefig(fp.replace('.pdf','.png'))
print(f"  ✅ Fig 1: {fp}"); plt.close()


# ── FIG 2: CVaR95 heatmap (with safe %) ──
fig, axes = plt.subplots(1, 3, figsize=(16, 8), sharey=True)
fig.suptitle('CVaR₉₅: Curriculum vs MSE (absolute Δ)\n'
             '(blue = curriculum better; "≈0" = both near-perfect hedging)',
             fontsize=13, fontweight='bold')

for pi, period in enumerate(PERIODS):
    ax = axes[pi]; rl = []; dm = []; annotations = []
    for opt in OPTION_NAMES:
        for kappa in KAPPA_LEVELS:
            os_ = 'Basket κ=' if 'basket' in opt else 'WoP κ='
            rl.append(f"{os_}{kappa:.2f}")
            row = []; ann_row = []
            for ds in DS_LIST:
                mv = collect(ds, opt, kappa, 'MSE', period, 'cvar95')
                cv = collect(ds, opt, kappa, 'CVaR_curriculum', period, 'cvar95')
                if mv and cv:
                    m_mse = np.mean(mv)
                    m_cur = np.mean(cv)
                    delta = (m_cur - m_mse) * 100  # in basis points of notional
                    pct = safe_pct(m_cur, m_mse)
                    row.append(delta)
                    if pct is not None:
                        ann_row.append(f'{pct:+.0f}%')
                    else:
                        ann_row.append('≈0')
                else:
                    row.append(0)
                    ann_row.append('—')
            dm.append(row)
            annotations.append(ann_row)

    data = np.array(dm)
    vmax = max(abs(data.min()), abs(data.max()), 1)
    im = ax.imshow(data, cmap='RdBu', aspect='auto', vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(3)); ax.set_xticklabels(DS_LIST)
    ax.set_title(SHORT_P[period], fontweight='bold')
    if pi == 0:
        ax.set_yticks(range(len(rl))); ax.set_yticklabels(rl)

    for i in range(len(rl)):
        for j in range(3):
            txt = annotations[i][j]
            c = 'white' if abs(data[i,j]) > vmax*0.4 else 'black'
            ax.text(j, i, txt, ha='center', va='center',
                    fontsize=8, color=c, fontweight='bold')

fig.colorbar(im, ax=axes.ravel().tolist(),
             label='Δ (bps of notional, negative = better)', shrink=0.6)
plt.tight_layout()
fp = os.path.join(FIG_FOLDER, 'fig2_cvar95_heatmap.pdf')
plt.savefig(fp); plt.savefig(fp.replace('.pdf','.png'))
print(f"  ✅ Fig 2: {fp}"); plt.close()


# ── FIG 3: Generator comparison (WoP, CVaR95) ──
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.suptitle('CVaR₉₅ by Generator — Worst-of Put (Historical Grand Avg)',
             fontsize=13, fontweight='bold')
opt = 'asian_worst_of_put'
for ki, kappa in enumerate(KAPPA_LEVELS):
    ax = axes[ki]; x = np.arange(3); w = 0.25
    for i, (lt, ll, col) in enumerate(zip(LOSS_TYPES, LOSS_LABELS, LOSS_COLORS)):
        means, ses = [], []
        for ds in DS_LIST:
            av = []
            for p in PERIODS:
                av.extend(collect(ds, opt, kappa, lt, p, 'cvar95'))
            means.append(max(np.mean(av), 0) if av else 0)
            ses.append(np.std(av)/np.sqrt(len(av)) if av else 0)
        ax.bar(x+i*w, means, w, label=ll, color=col, alpha=0.85,
               yerr=ses, capsize=3, edgecolor='white')
    ax.set_title(f'κ = {kappa:.2f}'); ax.set_xticks(x+w)
    ax.set_xticklabels(DS_LIST); ax.set_ylim(bottom=0)
    if ki == 0: ax.set_ylabel('CVaR₉₅'); ax.legend()
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fp = os.path.join(FIG_FOLDER, 'fig3_generator_cvar95_wop.pdf')
plt.savefig(fp); plt.savefig(fp.replace('.pdf','.png'))
print(f"  ✅ Fig 3: {fp}"); plt.close()


# ── FIG 4: Tradeoff scatter (Std vs CVaR95) ──
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.set_title('Std vs CVaR₉₅ Tradeoff: MSE → Curriculum\n'
             'Each point = 1 config, Historical Grand Avg',
             fontsize=11, fontweight='bold')

mkrs = {'GBM': 'o', 'Heston': 's', 'SBTS': '^'}
ocol = {'basket_asian_call': '#1976D2', 'asian_worst_of_put': '#D32F2F'}

for ds in DS_LIST:
    for opt in OPTION_NAMES:
        for kappa in KAPPA_LEVELS:
            ms_s, ms_c, cs_s, cs_c = [], [], [], []
            for p in PERIODS:
                ms_s.extend(collect(ds, opt, kappa, 'MSE', p, 'std'))
                ms_c.extend(collect(ds, opt, kappa, 'MSE', p, 'cvar95'))
                cs_s.extend(collect(ds, opt, kappa, 'CVaR_curriculum', p, 'std'))
                cs_c.extend(collect(ds, opt, kappa, 'CVaR_curriculum', p, 'cvar95'))
            if ms_s and cs_s:
                m1, m2 = np.mean(ms_s), max(np.mean(ms_c), 0)
                c1, c2 = np.mean(cs_s), max(np.mean(cs_c), 0)
                ax.annotate('', xy=(c1,c2), xytext=(m1,m2),
                           arrowprops=dict(arrowstyle='->', color='gray', alpha=0.4, lw=0.8))
                ax.scatter(m1, m2, marker=mkrs[ds], c='white',
                          edgecolors=ocol[opt], s=60, linewidths=1.5, zorder=5)
                ax.scatter(c1, c2, marker=mkrs[ds], c=ocol[opt],
                          edgecolors='black', s=60, linewidths=0.5, zorder=5)

legend_el = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='gray', ms=8, label='GBM'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor='gray', ms=8, label='Heston'),
    Line2D([0],[0], marker='^', color='w', markerfacecolor='gray', ms=8, label='SBTS'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='white',
           markeredgecolor='#1976D2', ms=8, label='Basket (MSE)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#1976D2',
           markeredgecolor='black', ms=8, label='Basket (Curric)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='white',
           markeredgecolor='#D32F2F', ms=8, label='WoP (MSE)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#D32F2F',
           markeredgecolor='black', ms=8, label='WoP (Curric)'),
]
ax.legend(handles=legend_el, loc='upper right', fontsize=8)
ax.set_xlabel('Std'); ax.set_ylabel('CVaR₉₅')
ax.set_ylim(bottom=0); ax.grid(alpha=0.3)
plt.tight_layout()
fp = os.path.join(FIG_FOLDER, 'fig4_tradeoff_scatter.pdf')
plt.savefig(fp); plt.savefig(fp.replace('.pdf','.png'))
print(f"  ✅ Fig 4: {fp}"); plt.close()


# ── FIG 5: Curriculum effect (Std + CVaR95) ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Curriculum Learning Effect by Period',
             fontsize=13, fontweight='bold')

for ai, (mk, mn) in enumerate([('std','Std'), ('cvar95','CVaR₉₅')]):
    ax = axes[ai]; x = np.arange(len(PERIODS)); w = 0.25
    for i, (lt, ll, col) in enumerate(zip(LOSS_TYPES, LOSS_LABELS, LOSS_COLORS)):
        vals = [max(grand_by_period(lt, p, mk), 0) for p in PERIODS]
        bars = ax.bar(x+i*w, vals, w, label=ll, color=col, alpha=0.85, edgecolor='white')
        for b, v in zip(bars, vals):
            label = f'{v:.4f}' if v >= ZERO_THRESHOLD else '≈0'
            ax.text(b.get_x()+b.get_width()/2, b.get_height() + 0.0003,
                    label, ha='center', va='bottom', fontsize=7)
    ax.set_title(mn); ax.set_xticks(x+w)
    ax.set_xticklabels(SHORT_PERIODS_DISPLAY, fontsize=9)
    ax.set_ylabel(mn); ax.set_ylim(bottom=0)
    if ai == 0: ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fp = os.path.join(FIG_FOLDER, 'fig5_curriculum_effect.pdf')
plt.savefig(fp); plt.savefig(fp.replace('.pdf','.png'))
print(f"  ✅ Fig 5: {fp}"); plt.close()


# ═══════════════════════════════════════════════════════════════
#  NEAR-ZERO DIAGNOSTIC
# ═══════════════════════════════════════════════════════════════
print(f"\n\n{'═' * 80}")
print("NEAR-ZERO DIAGNOSTIC — configs where |CVaR95| < threshold")
print(f"  Threshold: {ZERO_THRESHOLD}")
print(f"{'═' * 80}")

n_near_zero = 0
for p in PERIODS:
    for ds in DS_LIST:
        for opt in OPTION_NAMES:
            for kappa in KAPPA_LEVELS:
                for lt, ll in zip(LOSS_TYPES, LOSS_LABELS):
                    vals = collect(ds, opt, kappa, lt, p, 'cvar95')
                    if vals:
                        m = np.mean(vals)
                        if abs(m) < ZERO_THRESHOLD:
                            n_near_zero += 1
                            os_ = 'Basket' if 'basket' in opt else 'WoP'
                            if n_near_zero <= 20:
                                print(f"  {SHORT_P[p]:>8s}  {ds:<7s}  {os_:<7s}  "
                                      f"κ={kappa:.2f}  {ll:<20s}  CVaR95={m:+.4f}")

print(f"\n  Total near-zero entries: {n_near_zero}")
print(f"  → These should use absolute Δ in thesis tables, NOT %")


# ═══════════════════════════════════════════════════════════════
#  FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════
print(f"\n\n{'═' * 80}")
print("THESIS-READY NARRATIVE")
print(f"{'═' * 80}")

# Safe COVID-only %
m_covid_std = grand_by_period('MSE', 'COVID_2020', 'std')
c_covid_std = grand_by_period('CVaR_curriculum', 'COVID_2020', 'std')
m_covid_c95 = grand_by_period('MSE', 'COVID_2020', 'cvar95')
c_covid_c95 = grand_by_period('CVaR_curriculum', 'COVID_2020', 'cvar95')

print(f"\n  COVID-2020 (% meaningful here):")
print(f"    Std:    MSE {m_covid_std:.4f} → Curriculum {c_covid_std:.4f}  "
      f"({safe_pct(c_covid_std, m_covid_std):+.1f}%)")
print(f"    CVaR95: MSE {m_covid_c95:.4f} → Curriculum {c_covid_c95:.4f}  "
      f"({safe_pct(c_covid_c95, m_covid_c95):+.1f}%)")

# Grand (safe)
m_grand_std = np.mean([grand_by_period('MSE', p, 'std') for p in PERIODS])
c_grand_std = np.mean([grand_by_period('CVaR_curriculum', p, 'std') for p in PERIODS])
m_grand_c95 = np.mean([grand_by_period('MSE', p, 'cvar95') for p in PERIODS])
c_grand_c95 = np.mean([grand_by_period('CVaR_curriculum', p, 'cvar95') for p in PERIODS])

print(f"\n  Grand Average:")
print(f"    Std:    MSE {m_grand_std:.4f} → Curriculum {c_grand_std:.4f}  "
      f"({safe_pct(c_grand_std, m_grand_std):+.1f}%)")
print(f"    CVaR95: MSE {m_grand_c95:.4f} → Curriculum {c_grand_c95:.4f}  "
      f"({safe_pct(c_grand_c95, m_grand_c95):+.1f}%)")

print(f"\n  Key thesis statements (all using safe %):")
print(f"    1. Curriculum reduces COVID Std by {abs(safe_pct(c_covid_std, m_covid_std)):.0f}%")
print(f"    2. Curriculum reduces COVID CVaR95 by {abs(safe_pct(c_covid_c95, m_covid_c95)):.0f}%")
print(f"    3. Curriculum reduces grand CVaR95 by {abs(safe_pct(c_grand_c95, m_grand_c95)):.0f}%")
print(f"    4. Normal period: both MSE and curriculum achieve CVaR95 ≈ 0")
print(f"    5. CVaR95 winner count: Curriculum 45/54 (83%)")

print(f"\n\n  📁 Figures: {FIG_FOLDER}")
for f in sorted(os.listdir(FIG_FOLDER)):
    print(f"    {f} ({os.path.getsize(os.path.join(FIG_FOLDER, f))/1024:.0f} KB)")

print(f"\n{'═' * 80}")
print("DONE")
print(f"{'═' * 80}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
LOADING ALL RESULTS
  v2.5 GBM: 60 runs
  v2.5 Heston: 60 runs
  v2.5 SBTS: 60 runs

  v2.5 total: 180 (MSE + CVaR curriculum)
  v2.2 total: 360 (MSE + CVaR scratch)


════════════════════════════════════════════════════════════════════════════════
TABLE 1: GRAND SUMMARY — Absolute Values (all 18 configs averaged)
════════════════════════════════════════════════════════════════════════════════

  Std:
  Loss                      COVID  PostCOV   Normal    Grand
  ────────────────────────────────────────────────────────
  MSE                      0.0211   0.0129   0.0104   0.0148
  CVaR (scratch)           0.0204   0.0159   0.0125   0.0162
  CVaR (curriculum)        0.0193   0.0133   0.0107   0.0144

  CVaR95:
  Loss                      COVID  PostCOV   Normal    Grand
  ────────────────────────────────────────────────────────
  MSE                      0.072

/tmp/ipykernel_17412/615845901.py:387: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  ✅ Fig 2: /content/drive/MyDrive/PHD_SBTS/figures_v25/fig2_cvar95_heatmap.pdf
  ✅ Fig 3: /content/drive/MyDrive/PHD_SBTS/figures_v25/fig3_generator_cvar95_wop.pdf
  ✅ Fig 4: /content/drive/MyDrive/PHD_SBTS/figures_v25/fig4_tradeoff_scatter.pdf
  ✅ Fig 5: /content/drive/MyDrive/PHD_SBTS/figures_v25/fig5_curriculum_effect.pdf


════════════════════════════════════════════════════════════════════════════════
NEAR-ZERO DIAGNOSTIC — configs where |CVaR95| < threshold
  Threshold: 0.005
════════════════════════════════════════════════════════════════════════════════
    Normal  GBM      Basket   κ=0.95  MSE                   CVaR95=+0.0028
    Normal  GBM      Basket   κ=0.95  CVaR (curriculum)     CVaR95=+0.0010
    Normal  GBM      Basket   κ=1.00  MSE                   CVaR95=-0.0024
    Normal  GBM      WoP      κ=0.95  CVaR (curriculum)     CVaR95=+0.0027
    Normal  GBM      WoP      κ=1.00  CVaR (curriculum)     CVaR95=+0.0021
    Normal  GBM      WoP      κ=1.05  CVaR (curriculum)  

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Kiểm tra chính xác đường dẫn
path1 = '/content/drive/MyDrive/PHD_SBTS/historical_test_paths.npz'
path2 = '/content/drive/My Drive/PHD_SBTS/historical_test_paths.npz'

print(f"Path 1 exists: {os.path.exists(path1)}")
print(f"Path 2 exists: {os.path.exists(path2)}")

# Nếu cả 2 không tìm được, list folder
for base in ['/content/drive/MyDrive', '/content/drive/My Drive']:
    if os.path.exists(base):
        print(f"\n{base} exists. Looking for PHD_SBTS...")
        # Tìm folder chứa "PHD" hoặc "SBTS"
        for item in os.listdir(base):
            if 'PHD' in item.upper() or 'SBTS' in item.upper():
                full = os.path.join(base, item)
                print(f"  Found: {full}")
                if os.path.isdir(full):
                    for f in os.listdir(full):
                        if 'historical' in f.lower():
                            print(f"    ✅ {f}")

Mounted at /content/drive
Path 1 exists: True
Path 2 exists: True

/content/drive/MyDrive exists. Looking for PHD_SBTS...
  Found: /content/drive/MyDrive/PHD_SBTS
    ✅ historical_test_paths.npz
    ✅ fig_historical_overview.png
  Found: /content/drive/MyDrive/thesis_sbts

/content/drive/My Drive exists. Looking for PHD_SBTS...
  Found: /content/drive/My Drive/PHD_SBTS
    ✅ historical_test_paths.npz
    ✅ fig_historical_overview.png
  Found: /content/drive/My Drive/thesis_sbts


In [6]:
import numpy as np

data = np.load('/content/drive/MyDrive/PHD_SBTS/  ', allow_pickle=True)

print("Keys:", list(data.keys()))
print()

# Period names
if 'period_names' in data:
    periods = data['period_names']
    print("Periods:", periods)
    print()

# Check each period
for key in sorted(data.keys()):
    arr = data[key]
    print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")
    if 'S_norm' in key:
        print(f"    → {arr.shape[0]} paths × {arr.shape[1]} time steps × {arr.shape[2]} assets")
        print(f"    → Window size: {arr.shape[1]-1} trading days")

        # Check if paths overlap
        if arr.shape[0] > 1:
            # Compare starting points of consecutive paths
            starts = arr[:5, 0, :]  # first 5 paths, time 0
            print(f"    → First 5 starting values (asset 1): {starts[:, 0]}")
            print(f"    → All start at 1.0? {np.allclose(starts, 1.0)}")

print("\n" + "="*50)
print("SUMMARY")
print("="*50)
for key in sorted(data.keys()):
    if 'S_norm' in key:
        n_paths = data[key].shape[0]
        n_steps = data[key].shape[1] - 1
        period = key.replace('S_norm_', '')
        print(f"  {period}: {n_paths} paths of {n_steps} days")

Keys: ['S_norm_COVID_2020', 'S_norm_PostCOVID_2021_22', 'S_norm_Normal_2023_24', 'S_norm_all', 'S0', 'tickers', 'period_names', 'period_sizes']

Periods: ['COVID_2020' 'PostCOVID_2021_22' 'Normal_2023_24']

  S0: shape=(3,), dtype=float64
  S_norm_COVID_2020: shape=(253, 253, 3), dtype=float64
    → 253 paths × 253 time steps × 3 assets
    → Window size: 252 trading days
    → First 5 starting values (asset 1): [1. 1. 1. 1. 1.]
    → All start at 1.0? True
  S_norm_Normal_2023_24: shape=(500, 253, 3), dtype=float64
    → 500 paths × 253 time steps × 3 assets
    → Window size: 252 trading days
    → First 5 starting values (asset 1): [1. 1. 1. 1. 1.]
    → All start at 1.0? True
  S_norm_PostCOVID_2021_22: shape=(503, 253, 3), dtype=float64
    → 503 paths × 253 time steps × 3 assets
    → Window size: 252 trading days
    → First 5 starting values (asset 1): [1. 1. 1. 1. 1.]
    → All start at 1.0? True
  S_norm_all: shape=(1256, 253, 3), dtype=float64
    → 1256 paths × 253 time ste